# 02A — Processing AR cubes into corrected, analysis-ready cubes

This notebook is the **processing** step: it takes the raw cubes `01A_download_data.ipynb`
produced and writes corrected ones. It draws nothing — all plotting and analysis lives in
`03A_data_analisis.ipynb`, which reads what this notebook writes.

For every region (`data/raw/NOAA_<noaa>_<date>/region_01_*_cube.fits`) it:

1. Aligns the continuum / magnetogram / dopplergram cubes on their **common timestamps**.
2. Calibrates the dopplergram with the four Castellanos Durán et al. (2021) zero-point
   corrections, giving absolute m/s (`src/doppler_calibration.py`).
3. Segments each continuum frame into **umbra**, **penumbra** and **Qsun**, thresholding
   against that frame's own quiet-sun intensity.
4. Removes the quiet-sun background from the magnetogram by subtracting a fitted plane.
5. Builds the **hot spot** — the strong-field core inside the sunspot footprint.
6. Writes everything to `data/processed/NOAA_<noaa>_<date>/`.

## Outputs

| File | Contents |
| --- | --- |
| `region_01_continuum_cube.fits` | copied through, so downstream needs one directory only |
| `region_01_magnetogram_corrected_cube.fits` | quiet-sun plane removed |
| `region_01_dopplergram_calibrated_cube.fits` | absolute LOS velocity, m/s |
| `region_01_masks_cube.fits` | `uint8` bitmask — bit0 umbra, bit1 penumbra, bit2 hot spot |
| `region_01_qsun_means.fits` | per-frame quiet-sun mean B and v |

All carry the `TIMESTAMPS` extension and `HISTORY` cards recording which corrections were
applied, and all open directly in **DS9** as cubes with a frame slider.

**Caveats this notebook does *not* correct for**, and that you should not read past when
interpreting anything downstream:
- Mean magnetogram values are **signed**. If a box contains both polarities of a bipolar
  group they partially cancel.
- `B_los` changes purely geometrically as the region rotates; no `cos θ` correction is
  applied, so part of any magnetogram trend is projection, not field evolution.
- The umbra mask is *every* pixel below threshold in the box — other spots, pores and bad
  pixels included. There is no connected-component selection of the target spot.

**Requires cubes built by the current `make_cube`** — older ones have no `TIMESTAMPS`
extension and `read_cube` will refuse them rather than fall back to re-globbing the frame
directory, which cannot detect a missing mid-window frame.

In [2]:
import sys
import pathlib

project_root = pathlib.Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root.resolve()))

import numpy as np

from src.utilities import read_cube, write_cube
from src.doppler_calibration import calibrate_cube

DATA_DIR      = pathlib.Path('../data/raw')
PROCESSED_DIR = pathlib.Path('../data/processed')

# Segmentation thresholds as fractions of the *per-frame* quiet-sun continuum intensity.
# Absolute DN thresholds don't work here: quiet-sun continuum falls with limb darkening,
# so a fixed cut makes the mask areas drift systematically as the AR rotates, and for a
# near-limb AR the whole frame can fall under a fixed penumbra cut — penumbra becomes the
# entire frame and the quiet-sun mask becomes empty.
UMBRA_FRAC    = 0.60   # I < 0.60 I_qs -> umbra
PENUMBRA_FRAC = 0.90   # I < 0.90 I_qs (and not umbra) -> penumbra
QSUN_PERCENTILE = 80   # percentile of finite pixels used as the frame's I_qs estimate

# --- Hot spot ---------------------------------------------------------------
# The strong-field core inside the sunspot footprint: hot_spot = mag_filter(B) & (umbra|penumbra),
# matching src/sunspot_analysis.py's masks_from_cubes so the downstream metrics and plots
# treat it identically to the sebastian_sun_spots DS regions.
#
# The default is on |B| rather than signed B on purpose. The DS notebooks used `b > 500`
# or `b < -500` chosen per region, and getting the sign wrong yields an *empty* mask
# rather than an error — NOAA 11536's umbra averages about -400 G, so `b > 500` would
# select nothing at all there. Override per AR only when you deliberately want to isolate
# one polarity of a bipolar group.
HOTSPOT_THRESHOLD = 500          # gauss
hotspot_override  = {
    # 11536: lambda b: b < -500,   # negative polarity only
}

# --- Dopplergram zero point -------------------------------------------------
# CALIBRATE_DOPPLER applies the four physical corrections of Castellanos Durán et al.
# (2021) Sect. 2 — observatory velocity, large-scale flows, convective blueshift CLV,
# gravitational redshift — giving velocities on an absolute m/s scale.
#
# RESIDUAL_PLANE_FIT additionally fits and subtracts a plane over the quiet-sun pixels.
# That is the *empirical* correction this notebook used to rely on, and it is off by
# default on purpose: it flattens whatever gradient is left, which also throws away the
# absolute scale the physical corrections just established. Turn it on only to inspect
# what the physical model failed to remove, and don't read absolute velocities off the
# result when you do.
CALIBRATE_DOPPLER  = True
RESIDUAL_PLANE_FIT = False

# The magnetogram is unaffected by any of the above — those are Doppler zero-point
# effects — but still carries an instrumental offset, so it keeps the quiet-sun plane
# subtraction unconditionally.

# Mask bit assignments, also written into the FITS header of the mask cube.
BIT_UMBRA, BIT_PENUMBRA, BIT_HOTSPOT = 1, 2, 4

## Discover regions

Any `NOAA_*` directory with a `region_01_continuum_cube.fits` — this naturally skips the pre-HMI ARs (11039, 11041), which have no cube files at all.

In [3]:
regions = []
for region_dir in sorted(DATA_DIR.glob('NOAA_*')):
    cont_cube = region_dir / 'region_01_continuum_cube.fits'
    if cont_cube.exists():
        regions.append(region_dir)

print(f'{len(regions)} region(s) found:')
for r in regions:
    print(' ', r.name)

1 region(s) found:
  NOAA_11536_2012-07-31


## Per-region processing

`process_region` loads the three cubes and joins them **on their timestamps** (written into
each cube by `make_cube`) rather than trimming each to the shortest length. Individual JSOC
files do fail, and when a frame is missing mid-window — which the download logs show
happens — index-based trimming silently matches every later frame against a different time
in the other two series.

It then:

- calibrates the dopplergram through `calibrate_cube`, which reads the per-frame headers
  back from the frame directory (the tracked box sweeps across the sky and the `OBS_V*`
  keywords change every frame, so the corrections cannot be applied cube-wide);
- segments each continuum frame against that frame's *own* quiet-sun intensity
  (`UMBRA_FRAC`/`PENUMBRA_FRAC` × `I_qs`) instead of an absolute DN cut;
- subtracts a fitted **plane** from the magnetogram rather than a scalar mean, which takes
  out the gradient across the box and not just the uniform offset;
- builds the **hot spot** inside the sunspot footprint.

It returns the corrected cubes and masks. It computes no time series and draws nothing —
`03A_data_analisis.ipynb` derives all of that from the written cubes, so the two notebooks
can't drift apart.

In [3]:
def load_aligned(region_dir, calibrate=CALIBRATE_DOPPLER, v_sdo_by_time=None):
    """Load the three cubes and index them onto their common set of timestamps.

    The three series are downloaded independently and individual files do fail, so their
    frame counts differ. Trimming each to the shortest length only works if the missing
    frames are all at the *end* — when a frame is missing mid-window (which is what the
    download logs actually show) every later frame ends up matched against a different
    time in the other two series. Joining on the timestamps written into each cube by
    make_cube is exact regardless of where the gaps fall.

    When `calibrate` is set, the dopplergram is replaced by its physically calibrated
    version before the join. The corrections are per-frame (the tracked box sweeps across
    the sky and the OBS_V* keywords change every frame), so calibrate_cube reads the
    per-frame headers back from the region's frame directory.
    """
    cubes, times = {}, {}
    for name, fname in [('cont', 'continuum'), ('mag', 'magnetogram'), ('dop', 'dopplergram')]:
        data, ts = read_cube(region_dir / f'region_01_{fname}_cube.fits')
        cubes[name], times[name] = data.astype(np.float32), ts

    term_means = None
    if calibrate:
        corrected, dop_times, term_means = calibrate_cube(region_dir, v_sdo_by_time=v_sdo_by_time)
        if dop_times != times['dop']:
            raise ValueError('calibrate_cube returned different timestamps than the cube')
        cubes['dop'] = corrected

    common = sorted(set(times['cont']) & set(times['mag']) & set(times['dop']))
    if not common:
        raise ValueError(f'{region_dir.name}: the three series share no timestamps')

    dropped = {k: len(v) - len(common) for k, v in times.items()}
    if any(dropped.values()):
        print(f'  aligned on {len(common)} common frames; dropped {dropped} unmatched')

    for name in cubes:
        index = {t: i for i, t in enumerate(times[name])}
        cubes[name] = cubes[name][[index[t] for t in common]]

    if term_means is not None:
        keep = [i for i, t in enumerate(times['dop']) if t in set(common)]
        term_means = {k: v[keep] for k, v in term_means.items()}

    return cubes, common, term_means


def remove_quiet_sun_plane(frame, qsun_mask, xn, yn):
    """Fit a plane to the quiet-sun pixels and subtract it from the whole frame.

    Subtracting a scalar quiet-sun mean only removes the spatially uniform term — for the
    dopplergram that is mostly the SDO orbital velocity. What survives is the line-of-sight
    solar-rotation gradient across the box, and because the umbra sits off to one side of
    the box its mean picks up a residual that drifts as the region rotates. Removing a
    plane instead takes out that gradient to first order.

    This is the *empirical* alternative to the physical corrections in
    src/doppler_calibration.py. It always applies to the magnetogram (which those
    corrections don't address) but only to the dopplergram when RESIDUAL_PLANE_FIT is set.

    Returns (corrected_frame, coefficients) with coefficients = (offset, d/dx, d/dy) in
    the normalized coordinates xn, yn.
    """
    valid = qsun_mask & np.isfinite(frame)
    if valid.sum() < 3:
        return frame, np.array([np.nan, np.nan, np.nan])
    design = np.column_stack([np.ones(valid.sum()), xn[valid], yn[valid]])
    coef, *_ = np.linalg.lstsq(design, frame[valid].astype(np.float64), rcond=None)
    plane = coef[0] + coef[1] * xn + coef[2] * yn
    return (frame - plane).astype(np.float32), coef


def process_region(region_dir, noaa=None, v_sdo_by_time=None):
    """Correct one region's cubes and build its masks.

    Returns everything the writer and the summary need; it deliberately computes no plots
    and no time series — 03A_data_analisis.ipynb derives those from the written cubes.
    """
    cubes, timestamps, term_means = load_aligned(region_dir, v_sdo_by_time=v_sdo_by_time)
    cube_cont, cube_mag, cube_dop = cubes['cont'], cubes['mag'], cubes['dop']
    n_t, ny, nx = cube_cont.shape

    # Normalized pixel coordinates for the plane fit, so the design matrix stays well
    # conditioned regardless of box size.
    yy, xx = np.mgrid[0:ny, 0:nx]
    xn = ((xx - nx / 2) / (nx / 2)).astype(np.float64)
    yn = ((yy - ny / 2) / (ny / 2)).astype(np.float64)

    finite = np.isfinite(cube_cont)
    i_qs = np.array([np.percentile(cube_cont[t][finite[t]], QSUN_PERCENTILE)
                     if finite[t].any() else np.nan for t in range(n_t)])

    umbra    = (cube_cont < (UMBRA_FRAC * i_qs)[:, None, None]) & finite
    penumbra = (cube_cont < (PENUMBRA_FRAC * i_qs)[:, None, None]) & finite & ~umbra
    both     = umbra | penumbra
    qsun     = finite & ~both

    plane_coefs = {'mag': np.full((n_t, 3), np.nan), 'dop': np.full((n_t, 3), np.nan)}
    for t in range(n_t):
        cube_mag[t], plane_coefs['mag'][t] = remove_quiet_sun_plane(cube_mag[t], qsun[t], xn, yn)
        if RESIDUAL_PLANE_FIT:
            cube_dop[t], plane_coefs['dop'][t] = remove_quiet_sun_plane(cube_dop[t], qsun[t], xn, yn)

    # Hot spot on the *plane-corrected* magnetogram, so the threshold means the same thing
    # in every frame rather than drifting with the instrumental offset.
    mag_filter = hotspot_override.get(noaa, lambda b: np.abs(b) > HOTSPOT_THRESHOLD)
    hot_spot = mag_filter(cube_mag) & both

    qsun_means = {
        'mag': np.array([np.nanmean(cube_mag[t][qsun[t]]) if qsun[t].any() else np.nan
                         for t in range(n_t)]),
        'dop': np.array([np.nanmean(cube_dop[t][qsun[t]]) if qsun[t].any() else np.nan
                         for t in range(n_t)]),
    }

    masks = {'umbra': umbra, 'penumbra': penumbra, 'hot_spot': hot_spot, 'qsun': qsun}
    return dict(
        cubes={'cont': cube_cont, 'mag': cube_mag, 'dop': cube_dop},
        masks=masks, timestamps=timestamps, i_qs=i_qs, qsun_means=qsun_means,
        plane_coefs=plane_coefs, doppler_terms=term_means,
        area_px={name: m.reshape(n_t, -1).sum(axis=1) for name, m in masks.items()},
    )

In [4]:
from astropy.io import fits


def region_noaa(region_dir):
    """NOAA number from a `NOAA_<number>_<date>` directory name, or None."""
    parts = region_dir.name.split('_')
    return int(parts[1]) if len(parts) > 1 and parts[1].isdigit() else None


def write_region(region_dir, result):
    """Write one region's corrected cubes, masks and quiet-sun means.

    Everything goes out through src.utilities.write_cube so it comes back through
    read_cube unchanged, and so DS9 sees the same structure as the raw cubes: a 3D
    primary HDU carrying the reference frame's spatial WCS, plus a TIMESTAMPS extension.
    """
    out_dir = PROCESSED_DIR / region_dir.name
    timestamps = result['timestamps']

    # Reuse the raw continuum cube's header so the corrected cubes keep the spatial WCS,
    # and DS9 puts them on the same footprint as the frames they came from.
    src_header = fits.getheader(region_dir / 'region_01_continuum_cube.fits')
    for key in ('NAXIS', 'NAXIS1', 'NAXIS2', 'NAXIS3', 'NFRAMES', 'BITPIX'):
        src_header.pop(key, None)

    custom_hotspot = region_noaa(region_dir) in hotspot_override
    provenance = [
        f'02A: {len(timestamps)} frames, aligned on common timestamps',
        f'02A: umbra < {UMBRA_FRAC} I_qs, penumbra < {PENUMBRA_FRAC} I_qs (I_qs = p{QSUN_PERCENTILE})',
        '02A: hot spot from a per-AR hotspot_override filter' if custom_hotspot else
        f'02A: hot spot |B| > {HOTSPOT_THRESHOLD} G within the sunspot',
    ]
    doppler_history = (
        ['02A: dopplergram calibrated - Castellanos Duran 2021 sdo+lsf+clv+gravity']
        if CALIBRATE_DOPPLER else ['02A: dopplergram NOT calibrated'])
    if RESIDUAL_PLANE_FIT:
        doppler_history.append('02A: residual quiet-sun plane also subtracted (absolute scale lost)')

    written = {}
    written['continuum'] = write_cube(
        result['cubes']['cont'], out_dir / 'region_01_continuum_cube.fits',
        header=src_header, timestamps=timestamps,
        history=provenance + ['02A: continuum copied through uncorrected'])

    written['magnetogram'] = write_cube(
        result['cubes']['mag'], out_dir / 'region_01_magnetogram_corrected_cube.fits',
        header=src_header, timestamps=timestamps,
        history=provenance + ['02A: quiet-sun plane subtracted from the magnetogram'])

    written['dopplergram'] = write_cube(
        result['cubes']['dop'], out_dir / 'region_01_dopplergram_calibrated_cube.fits',
        header=src_header, timestamps=timestamps, history=provenance + doppler_history)

    # Masks as one uint8 bitmask: DS9 renders integers far more cleanly than floats, and a
    # single file keeps the overlapping hot spot alongside the regions it sits inside.
    # The bit values and threshold fractions are keywords rather than a convention, so
    # load_noaa_region reads back exactly what was used instead of assuming defaults.
    m = result['masks']
    bitmask = (m['umbra'].astype(np.uint8) * BIT_UMBRA
               | m['penumbra'].astype(np.uint8) * BIT_PENUMBRA
               | m['hot_spot'].astype(np.uint8) * BIT_HOTSPOT)
    mask_header = src_header.copy()
    mask_header['BUNIT']    = ('', 'bit flags, see BIT_* keywords')
    mask_header['BIT_UMB']  = (BIT_UMBRA, 'bit value for umbra')
    mask_header['BIT_PEN']  = (BIT_PENUMBRA, 'bit value for penumbra')
    mask_header['BIT_HOT']  = (BIT_HOTSPOT, 'bit value for hot spot')
    mask_header['UMB_FRAC'] = (UMBRA_FRAC, 'umbra threshold as a fraction of I_qs')
    mask_header['PEN_FRAC'] = (PENUMBRA_FRAC, 'penumbra threshold as a fraction of I_qs')
    written['masks'] = write_cube(
        bitmask, out_dir / 'region_01_masks_cube.fits',
        header=mask_header, timestamps=timestamps,
        history=provenance + ['02A: 0 = quiet sun; bits overlap (hot spot is inside the spot)'])

    # Quiet-sun means as a small table. compute_metrics only ever takes their per-frame
    # mean, so saving the numbers avoids carrying two more full-size cubes around.
    cols = [
        fits.Column(name='T_OBS', format='23A',
                    array=np.array([t.isoformat() for t in timestamps])),
        fits.Column(name='MEAN_MAG_QSUN', format='E', array=result['qsun_means']['mag']),
        fits.Column(name='MEAN_DOP_QSUN', format='E', array=result['qsun_means']['dop']),
        fits.Column(name='I_QS', format='E', array=result['i_qs']),
    ]
    qsun_path = out_dir / 'region_01_qsun_means.fits'
    qsun_path.parent.mkdir(parents=True, exist_ok=True)
    fits.BinTableHDU.from_columns(cols, name='QSUN').writeto(qsun_path, overwrite=True)
    written['qsun_means'] = str(qsun_path)

    return out_dir, written

In [5]:
results = {}
for region_dir in regions:
    print(region_dir.name)
    noaa = region_noaa(region_dir)
    result = process_region(region_dir, noaa=noaa)
    out_dir, written = write_region(region_dir, result)
    results[region_dir.name] = result

    n_t = len(result['timestamps'])
    area = result['area_px']
    span_h = (result['timestamps'][-1] - result['timestamps'][0]).total_seconds() / 3600
    print(f'  {n_t} frames over {span_h:.1f} h, box {result["cubes"]["cont"].shape[1:]} px')
    print(f'  mean area px  umbra={area["umbra"].mean():.0f}  penumbra={area["penumbra"].mean():.0f}  '
          f'hot spot={area["hot_spot"].mean():.0f}  quiet sun={area["qsun"].mean():.0f}')
    # An empty hot spot is the failure mode to watch for: a polarity-specific override
    # silently selects nothing when the spot has the other sign.
    empty = int((area['hot_spot'] == 0).sum())
    if empty:
        print(f'  WARNING: hot spot is empty in {empty}/{n_t} frame(s) — check the filter '
              f'for NOAA {noaa} against the actual polarity')
    print(f'  quiet sun  <B>={np.nanmean(result["qsun_means"]["mag"]):7.1f} G   '
          f'<v>={np.nanmean(result["qsun_means"]["dop"]):7.1f} m/s')
    print(f'  -> {out_dir}')
    for name, path in written.items():
        print(f'       {name:12s} {pathlib.Path(path).name}')

NOAA_11536_2012-07-31


  aligned on 359 common frames; dropped {'cont': 2, 'mag': 2, 'dop': 0} unmatched


  359 frames over 72.0 h, box (139, 484) px
  mean area px  umbra=485  penumbra=2459  hot spot=1532  quiet sun=64333
  quiet sun  <B>=   -0.0 G   <v>= -229.3 m/s
  -> ../data/processed/NOAA_11536_2012-07-31
       continuum    region_01_continuum_cube.fits
       magnetogram  region_01_magnetogram_corrected_cube.fits
       dopplergram  region_01_dopplergram_calibrated_cube.fits
       masks        region_01_masks_cube.fits
       qsun_means   region_01_qsun_means.fits


## Diagnostics — was the signal solar or instrumental?

A numeric summary of what was removed and what the segmentation did. The corresponding
plots live in `03A_data_analisis.ipynb`; what matters here is that nothing looks absurd
before the cubes get written.

- **`I_qs`** should drift smoothly and slowly (limb darkening). The point of thresholding
  against it is that the mask areas should *not* inherit that drift, so a large `I_qs`
  swing next to a flat area is the correct outcome, not a contradiction.
- **Mask areas** should be roughly flat. A monotonic ramp means the segmentation is still
  tracking limb darkening rather than the spot; a step means frames of mixed provenance.
- **Doppler terms** are what the old empirical plane fit used to absorb blindly. `sdo`
  swings by hundreds of m/s over a day; if it doesn't, the per-frame headers aren't being
  read.

In [4]:
def summarize(region_name, result):
    n_t = len(result['timestamps'])
    print(f'\n{region_name}  ({n_t} frames)')

    i_qs = result['i_qs']
    print(f'  I_qs           {np.nanmin(i_qs):8.0f} .. {np.nanmax(i_qs):8.0f} DN   '
          f'({100 * np.ptp(i_qs) / np.nanmean(i_qs):.1f}% drift)')

    print('  mask area (px)      mean     min     max   drift')
    for name, area in result['area_px'].items():
        drift = 100 * np.ptp(area) / area.mean() if area.mean() else np.nan
        print(f'    {name:12s} {area.mean():8.0f} {area.min():7.0f} {area.max():7.0f} {drift:6.1f}%')

    terms = result['doppler_terms']
    if terms:
        print('  Doppler terms removed (m/s)     mean   peak-to-peak')
        for name, series in terms.items():
            print(f'    {name:10s} {np.nanmean(series):20.1f} {np.ptp(series):10.1f}')

    coefs = result['plane_coefs']['mag']
    if np.isfinite(coefs).any():
        grad = np.hypot(coefs[:, 1], coefs[:, 2])
        print(f'  magnetogram plane |gradient|  {np.nanmean(grad):.1f} G per half-box')


for region_name, result in results.items():
    summarize(region_name, result)

NameError: name 'results' is not defined